Funathon NACE code classifier
URL: https://aiml4os.github.io/funathon-project2/1-ttc.html

*1. Install the dependencies*
All required packages are listed in pyproject.toml. Run the following command once at the root of the project to install them into a virtual environment:

In [ ]:
uv sync

*1.3. Start an MLflow service*

Launch an MLflow service from the SSP Cloud service catalogue. To do so, go on SSPCloud, click on My Services, then on New Service, then on Automation, then on MLflow, then on Launch. Once the service starts, its public URL appears in the service details. It typically looks like https://user-<username>-mlflow.user.lab.sspcloud.fr.

Create a file named .env at the root of the project and fill it in as follows:

MLFLOW_TRACKING_URI=<add the MLflow server URL>
MLFLOW_TRACKING_USERNAME=<add your username>
MLFLOW_TRACKING_PASSWORD=<add your password>

*2 Load the data*
The dataset consists of synthetic labelled examples generated for the NACE rev 2.1 nomenclature. Each row contains a short text description (label) and the corresponding NACE code (code).

*2.1 Question 1 — Import libraries and load environment variables*
Import the package mlflow and the load_dotenv from the dotenv package. Then execute load_dotenv(override=True) to load your .env file.

In [12]:
import os
import mlflow
from dotenv import load_dotenv

load_dotenv(override=True)

/opt/python/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

*2.2 Question 2 — Load the dataset from s3*
Use Polars to load the parquet file directly from this public URL:

https://minio.lab.sspcloud.fr/projet-formation/diffusion/funathon/2026/project2/generation_None_temp08.parquet
Print the first rows and the total number of observations. Do you understand all columns?

In [15]:
import polars as pl

df = pl.read_parquet("https://minio.lab.sspcloud.fr/projet-formation/diffusion/funathon/2026/project2/generation_None_temp08.parquet")

In [19]:
print(df.head(10))

shape: (10, 3)
┌───────┬─────────────────────────────────┬─────────────────────────────────┐
│ code  ┆ name                            ┆ label                           │
│ ---   ┆ ---                             ┆ ---                             │
│ str   ┆ str                             ┆ str                             │
╞═══════╪═════════════════════════════════╪═════════════════════════════════╡
│ 01.11 ┆ Growing of cereals, other than… ┆ Pulses cultivation for market   │
│ 01.11 ┆ Growing of cereals, other than… ┆ Legume crop production activit… │
│ 01.11 ┆ Growing of cereals, other than… ┆ Broad bean farming operations   │
│ 01.11 ┆ Growing of cereals, other than… ┆ Chickpea harvesting and proces… │
│ 01.11 ┆ Growing of cereals, other than… ┆ Production of dried beans and … │
│ 01.11 ┆ Growing of cereals, other than… ┆ Commercial cultivation of lent… │
│ 01.11 ┆ Growing of cereals, other than… ┆ Cowpea yield and quality contr… │
│ 01.11 ┆ Growing of cereals, other than… ┆ Pigeo

In [17]:
print(f"Total rows: {len(df)}")

Total rows: 70000


*2.3 Question 3 — Count unique NACE codes*
How many unique NACE codes are present in the dataset? Store this number in a variable called n_classes. This number will define the number of output classes the model must predict.

In [18]:
n_classes = df['code'].n_unique()
print(f"Number of unique NACE codes: {n_classes}")

Number of unique NACE codes: 311


*3 Split the data*
Before training a model, we need to split the dataset into three subsets:

- Train: the examples the model learns from.
- Validation: used during training to monitor performance and trigger early stopping (i.e. stop training before overfitting).
- Test: held out until the very end to give an unbiased estimate of the model’s performance on unseen data.

Why three splits and not two?
Using the validation set to tune hyperparameters or stop training means it indirectly influences the model. The test set must therefore remain completely untouched during training so that the final evaluation is truly unbiased.

*3.1 Question 1 — Split the dataset into train / validation / test sets*
Use train_test_split from sklearn.model_selection to split the dataset into train, validation, and test sets (70% / 15% / 15%). Do not forget to choose a random_state. Separate the target y from the features X, and convert them to numpy arrays. You should obtain objects X_train, y_train, and so on.

In [20]:
from sklearn.model_selection import train_test_split

train_df, tmp_df = train_test_split(df, test_size=0.30, random_state=42)
val_df, test_df  = train_test_split(tmp_df, test_size=0.50, random_state=42)

X_train, y_train = train_df["label"].to_numpy(), train_df["code"].to_numpy()
X_val, y_val = val_df["label"].to_numpy(), val_df["code"].to_numpy()
X_test, y_test = test_df["label"].to_numpy(), test_df["code"].to_numpy()

In [21]:

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

Train: 49000 | Val: 10500 | Test: 10500


*3.2 Question 2 — Encode the labels*
The code column contains strings like "47.11A" (codes from the NACE classification). Neural networks do not understand strings and require integer targets (1, 2, 3…), so we need to map each unique NACE code to an integer index. To do that, you will use an encoder.

Use LabelEncoder from sklearn.preprocessing (see here for the documentation). Call .fit() on the code column of your training dataframe (after having converted it to a numpy array with .to_numpy()).

We use LabelEncoder to map the categorical target y to integers 0, 1, …, n_classes - 1, the format expected by scikit-learn classifiers.

Fit the LabelEncoder on the training set only — fitting on validation or test data would be data leakage. However, this means any NACE code absent from the training set will cause an error at inference time. Make sure every code appears at least once in training before splitting.

In [26]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()
encoder.fit(train_df['code'].to_numpy())

LabelEncoder()

In [27]:
all_codes  = set(df['code'])
train_codes = set(train_df['code'])
missing = all_codes - train_codes

if missing:
    print(f"WARNING: {len(missing)} code(s) missing from training set: {missing}")
else:
    print(f"OK — all {len(all_codes)} codes appear in the training set.")

OK — all 311 codes appear in the training set.


*3.3 Question 3 — Prepare the labels to use them with ttc*
torchTextClassifiers requires labels to be wrapped in a ValueEncoder object. This is straightforward: you just pass your already-fitted LabelEncoder to the ValueEncoder constructor.

Import the ValueEncoder class from the torchTextClassifiers.value_encoder subpackage, then create a value_encoder object by passing your fitted label_encoder as an argument.

While LabelEncoder plays the role of the “converter” from string labels to integers, working with raw integers at prediction time can be cumbersome — you have to manually map predictions back to their original label names. The ValueEncoder solves this by integrating tightly with the torchTextClassifiers ecosystem: when you pass it to a model, predictions are automatically returned as human-readable labels rather than integers. As a bonus, ValueEncoder also supports additional categorical input features — advanced readers can find examples here. https://inseefrlab.github.io/torchTextClassifiers/tutorials/mixed_features.html

In [ ]:
%pip install git+https://github.com/InseeFrLab/torchTextClassifiers.git

In [30]:
from torchTextClassifiers.value_encoder import ValueEncoder

value_encoder = ValueEncoder(label_encoder=encoder)